# DATA 622: Homework 5
**Author:** Brett Allen

**Date Completed:** TBD (WIP)

## Setup

In [1]:
!python -m pip install -q nltk "transformers==5.3.0" torch scikit-learn networkx beautifulsoup4 requests

Verify pip module versions

In [2]:
%%bash
for module in nltk transformers torch scikit-learn networkx beautifulsoup4 requests; do echo "====="; echo $module; pip show $module | grep Version; done
echo "====="

=====
nltk
Version: 3.9.3
License: Apache License, Version 2.0
=====
transformers
Version: 5.3.0
=====
torch
Version: 2.10.0
=====
scikit-learn
Version: 1.8.0
=====
networkx
Version: 3.6.1
=====
beautifulsoup4
Version: 4.14.3
=====
requests
Version: 2.32.5
=====


### Imports

In [32]:
import requests
import nltk
import numpy as np
import networkx as nx
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    pipeline, 
    AutoModelForCausalLM, 
    AutoTokenizer,
    BartTokenizer, 
    BartForConditionalGeneration,
    AutoConfig
)

### Configurations

In [4]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/brett/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/brett/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Questions
Use the following article:

https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/.

In [5]:
url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

In [6]:
# Use user agent to avoid potential blocking by the website
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
response = requests.get(url, headers=headers)
response.status_code

200

In [7]:
# Parse the HTML content with beautifulsoup
soup = BeautifulSoup(response.text, "html.parser")

# Extract the article text (paragraphs)
paragraphs = soup.find_all("p")
article_text = " ".join([p.get_text() for p in paragraphs])

In [8]:
# Print the first 500 characters of the article text
print(article_text[:500] + "...")

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell...


In [9]:
# Convert article text into sentences
sentences = sent_tokenize(article_text)
print(f"Total sentences: {len(sentences)}")

Total sentences: 16


In [10]:
# Inspect first 5 sentences
print(sentences[:5])

['Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.', 'Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."', '"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?', 'Because I sure as hell hope that it is not your testimony," Turner dug in.', '"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.']


### 1. Extract and print an extractive summary using the TextRank method.

In [11]:
# # Vectorize sentences using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(sentences)

In [12]:
# Create similarity matrix to prepare for TextRank
similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

In [13]:
# Apply TextRank algorithm using PageRank on the similarity graph
nx_graph = nx.from_numpy_array(similarity_matrix)
scores = nx.pagerank(nx_graph)

In [14]:
scores

{0: 0.07743182158702094,
 1: 0.07220737882730344,
 2: 0.07833435678381329,
 3: 0.05643920408603973,
 4: 0.0648499702801737,
 5: 0.07258043540953774,
 6: 0.05974219806047407,
 7: 0.060321335161830886,
 8: 0.061283536198635524,
 9: 0.06138285157193014,
 10: 0.009900990099009903,
 11: 0.07159118768738873,
 12: 0.057585961548067485,
 13: 0.06461670576290306,
 14: 0.07265646036971052,
 15: 0.05907560656616065}

In [15]:
# Rank sentences based on TextRank scores (sort in descending order so that the highest scored sentences are first)
ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

In [16]:
# Print top 5 ranked sentences as the summary
summary = " ".join([ranked_sentences[i][1] for i in range(5)])

print("\nTextRank Summary:\n")
print(summary)


TextRank Summary:

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Greenland belongs to the Greenlanders," Danish Prime Minister Mette Frederiksen said after Vance's visit. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


### 2. Extract and print an extractive summary using a frequency-based sentence scoring method.

In [17]:
# Tokenize article text into words and create frequency table for frequency-based sentence scoring method
words = word_tokenize(article_text.lower())

freq_table = {}
for word in words:
    # Only consider alphabetic words (ignore punctuation and numbers)
    if word.isalpha():
        freq_table[word] = freq_table.get(word, 0) + 1

In [18]:
len(freq_table)

153

In [19]:
sentence_scores = {}

# Iterate through sentences and score them based on word frequencies (sum of frequencies of words in the sentence)
for sentence in sentences:
    for word in word_tokenize(sentence.lower()):
        # If the word is in the frequency table, add its frequency to the sentence score
        if word in freq_table:
            sentence_scores[sentence] = sentence_scores.get(sentence, 0) + freq_table[word]

In [20]:
len(sentence_scores)

16

In [21]:
# Inspect sentence scores
sentence_scores

{'Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.': 90,
 'Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."': 122,
 '"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?': 81,
 'Because I sure as hell hope that it is not your testimony," Turner dug in.': 40,
 '"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.': 79,
 'President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won\'t be necessary.': 92,
 'He has insisted that acquiring Greenland is necessary for national security, citing growing Chinese and Russian influence in the region.': 70,
 'The

In [22]:
# Rank sentences based on frequency scores (sort in descending order so that the highest scored sentences are first)
ranked_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)

In [23]:
# Create summary by joining the top 5 ranked sentences
summary = " ".join(ranked_sentences[:5])

print("\nFrequency-Based Summary:\n")
print(summary)


Frequency-Based Summary:

During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force. In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." The island is also rich in critical minerals that the U.S. wants to challenge Chinese monopolies in some industries, USA TODAY has reported. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.


### 3. Generate and print an abstractive summary using a pretrained transformer model (e.g., BART or T5).

In [ ]:
model_name = "facebook/bart-large-cnn"

config = AutoConfig.from_pretrained(model_name)

# Get the tokenizer max length
# Common attribute names depending on the model architecture:
max_length = getattr(config, "max_position_embeddings", None) or \
             getattr(config, "n_positions", None)

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 2046.29it/s]


In [30]:
print(f'Max length for {model_name}: {max_length}')

Max length for facebook/bart-large-cnn: 1024


In [33]:
# BART has a ~1024 token limit so truncate long articles
inputs = tokenizer(article_text, return_tensors="pt", max_length=max_length, truncation=True)

# Generate summary with BART using beam search and length penalty to encourage longer summaries
summary_ids = model.generate(
    inputs["input_ids"],   # Input IDs for the article text.
    max_length=200,        # Maximum number of tokens in the summary.
    min_length=80,         # Minimum number of tokens before generation can stop.
    length_penalty=1.5,    # Encourages longer summaries when > 1.
    num_beams=4,           # Improves summary quality via beam search.
    early_stopping=True    # Stop generation when at least num_beams sentences are finished per batch.
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("\nAbstractive Summary (BART):\n")
print(summary)


Abstractive Summary (BART):

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. He has insisted that acquiring Greenland is necessary for national security, citing growing Chinese and Russian influence in the region.


### 4. Print a Lead-3 summary (the first three sentences of the article).

In [34]:
lead3_summary = " ".join(sentences[:3])

print("\nLead-3 Summary:\n")
print(lead3_summary)


Lead-3 Summary:

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?


### 5. Print a manual compression summary, limiting the result to about 20% of the original sentences.

In [35]:
# Determine number of sentences to include in summary (20% of total sentences)
num_sentences = int(len(sentences) * 0.2)
print(f"\nNumber of sentences in compressed summary (20% of total): {num_sentences}")


Number of sentences in compressed summary (20% of total): 3


In [36]:
compressed_summary = " ".join(sentences[:num_sentences])

print("\nManual Compression Summary (~20%):\n")
print(compressed_summary)


Manual Compression Summary (~20%):

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?


### 6. Use an LLM to summary the text.

In [2]:
# Use a large language model (Qwen3.5-2B) - see https://huggingface.co/Qwen/Qwen3.5-2B
model_name = "Qwen/Qwen3.5-2B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
# Create a prompt to have the LLM summarize the article text
prompt = f"Summarize the following article:\n\n{article_text}\n\nSummary:"

In [ ]:
# Determine max length for the model input (dynamically)
max_length = tokenizer.model_max_length
print(f"Model max input length: {max_length} tokens")

In [ ]:
# Tokenize the input text
inputs = tokenizer(prompt, return_tensors="pt", truncation=False, max_length=2048)

In [ ]:
# Generate the summary
output_sequences = model.generate(
    input_ids=inputs["input_ids"], 
    max_new_tokens=max_length,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    pad_token_id=tokenizer.eos_token_id
)

# Decode the generated tokens to text
summary = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

# Post-process the output to isolate the summary part
# This might require fine-tuning the prompt and post-processing based on model behavior
summary_start = summary.find("Summary:") + len("Summary:")
summary = summary[summary_start:].strip()